In [1]:
import pickle
import numpy as np

import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model

I0000 00:00:1788371466.630099   35473 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788371467.958684   35473 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788371472.730620   35473 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
model = load_model("model.keras")
model.summary()

E0000 00:00:1788371479.773599   35473 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,013 (168.02 KB)

 Trainable params: 14,337 (56.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 28,676 (112.02 KB)

In [3]:
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

In [4]:
data = pd.read_csv("Dataset/creditcard.csv")
print(data.shape)

(284807, 31)


In [5]:
data = data.drop_duplicates()
print(data.shape)

(283726, 31)


In [6]:
X = data.drop("Class", axis=1)
y = data["Class"]

In [7]:
index = 0
transaction = X.iloc[index : index + 1]
actual_class = y.iloc[index]
print("Actual class:", "FRAUD" if actual_class == 1 else "NORMAL")

Actual class: NORMAL


In [8]:
transaction_scaled = scaler.transform(transaction)

In [9]:
probability = model.predict(transaction_scaled, verbose=0)[0][0]
threshold = 0.5
prediction = int(probability >= threshold)
print(f"Fraud probability: " f"{probability * 100:.4f}%")
if prediction == 1:
    print("Prediction: FRAUD!!!")
else:
    print("Prediction: NORMAL...")

Fraud probability: 4.2848%
Prediction: NORMAL...


In [10]:
for index in range(10):
    transaction = X.iloc[index : index + 1]
    actual = y.iloc[index]
    transaction_scaled = scaler.transform(transaction)
    probability = model.predict(transaction_scaled, verbose=0)[0][0]
    prediction = int(probability >= 0.5)
    print(
        f"Transaction {index + 1}: "
        f"Prediction="
        f"{'FRAUD' if prediction else 'NORMAL'}, "
        f"Probability={probability * 100:.2f}%, "
        f"Actual="
        f"{'FRAUD' if actual else 'NORMAL'}"
    )

Transaction 1: Prediction=NORMAL, Probability=4.28%, Actual=NORMAL
Transaction 2: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 3: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 4: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 5: Prediction=NORMAL, Probability=0.11%, Actual=NORMAL
Transaction 6: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 7: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 8: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 9: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
Transaction 10: Prediction=NORMAL, Probability=0.00%, Actual=NORMAL
